# Chapter 1 — Exercises: Worked Solutions

Complete worked solutions for Chapter 1 (Introduction to Equations of State).

**Exercises:**
1. Compressibility factor and ideal gas deviation
2. Critical properties and acentric factor
3. Industrial EoS selection criteria

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314  # J/(mol K)

---
## Exercise 1.1 — Compressibility Factor of Methane

**Problem:** Calculate the compressibility factor $Z = Pv/RT$ of methane
at $T = 300$ K for pressures from 1 to 200 bar using NeqSim (SRK EoS).
Plot $Z$ vs $P$ and identify the pressure range where the ideal gas law
gives less than 5% error.

### Solution

The compressibility factor $Z$ quantifies the deviation from ideal gas
behavior ($Z = 1$). For methane at moderate temperatures, $Z < 1$ at
intermediate pressures (attractive forces dominate) and $Z > 1$ at very
high pressures (repulsive forces dominate).

In [3]:
if NEQSIM_MODE == "pip":
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

pressures = np.arange(1, 205, 5)
Z_srk = []
T_K = 300.0

for P in pressures:
    f = SystemSrkEos(T_K, float(P))
    f.addComponent("methane", 1.0)
    f.setMixingRule("classic")
    ops = ThermodynamicOperations(f)
    ops.TPflash()
    f.initProperties()
    Z_srk.append(float(f.getPhase(0).getZ()))

# Find range where |Z - 1| < 0.05
ideal_limit = max(p for p, z in zip(pressures, Z_srk) if abs(z - 1.0) < 0.05)
print(f"Ideal gas approximation (<5% error) valid up to ~{ideal_limit} bar")

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures, Z_srk, color=BLUE, linewidth=1.5, label="SRK")
ax.axhline(y=1.0, color="grey", linestyle=":", alpha=0.5, label="Ideal gas")
ax.fill_between(pressures, 0.95, 1.05, alpha=0.1, color=GREEN, label="$\\pm 5\\%$")
ax.set_xlabel("Pressure (bar)"); ax.set_ylabel("$Z = Pv/RT$")
ax.set_title("Exercise 1.1: Methane compressibility at 300 K")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch01_ex01_Z_methane.png")

Ideal gas approximation (<5% error) valid up to ~31 bar
Saved: fig_ch01_ex01_Z_methane.png


**Answer:** At 300 K, the ideal gas law is accurate to within 5% up to
approximately 20–30 bar for methane. Above this pressure, attractive
intermolecular interactions cause significant deviation ($Z < 1$),
reaching a minimum around 100–150 bar before repulsive interactions
begin to dominate at higher pressures.

---
## Exercise 1.2 — Critical Properties and the Acentric Factor

**Problem:** Using NeqSim, compute the vapor pressure of $n$-hexane at
$T_r = 0.7$ (i.e., $T = 0.7 \times T_c$). From this, calculate the
acentric factor:

$$\omega = -\log_{10}\left(\frac{P_{\text{vap}}}{P_c}\right) - 1$$

Compare with the literature value $\omega_{\text{hexane}} = 0.301$.

In [4]:
# Critical properties of n-hexane
Tc_hex = 507.6  # K
Pc_hex = 30.25  # bar
omega_lit = 0.301

T_07 = 0.7 * Tc_hex
print(f"T_r = 0.7 => T = {T_07:.1f} K = {T_07 - 273.15:.1f} C")

f = SystemSrkEos(T_07, 1.0)
f.addComponent("n-hexane", 1.0)
f.setMixingRule("classic")
ops = ThermodynamicOperations(f)
ops.bubblePointPressureFlash(False)

Pvap = float(f.getPressure("bara"))
omega_calc = -np.log10(Pvap / Pc_hex) - 1.0

print(f"P_vap at T_r = 0.7: {Pvap:.3f} bar")
print(f"Calculated omega: {omega_calc:.4f}")
print(f"Literature omega:  {omega_lit:.4f}")
print(f"Relative error: {abs(omega_calc - omega_lit)/omega_lit*100:.1f}%")

T_r = 0.7 => T = 355.3 K = 82.2 C
P_vap at T_r = 0.7: 1.511 bar
Calculated omega: 0.3016
Literature omega:  0.3010
Relative error: 0.2%


**Answer:** The SRK EoS reproduces the acentric factor of $n$-hexane
within a few percent, confirming that the $\alpha(T)$ function is well
calibrated for hydrocarbon vapor pressures. The acentric factor captures
the non-sphericity and polarity of the molecule.

---
## Exercise 1.3 — EoS Selection Criteria

**Problem:** For each of the following systems, recommend an equation
of state and explain your reasoning:

1. Dry natural gas pipeline ($>95\%$ methane, $T > 0^\circ$C, $P < 100$ bar)
2. Wet gas with glycol dehydration
3. CO$_2$ capture with aqueous amine solution
4. Crude oil with water cut

### Solution

| System | Recommended EoS | Reason |
|--------|----------------|--------|
| Dry natural gas | SRK or PR | Non-polar, well-characterized components |
| Wet gas + glycol | CPA | Glycol is strongly hydrogen-bonding |
| CO$_2$ + amines | e-CPA or electrolyte NRTL | Ions and strong chemical solvation |
| Crude + water | CPA | Water–hydrocarbon mutual solubility |

The key decision criterion is whether **association** (hydrogen bonding)
or **electrolyte** effects are significant. For non-polar hydrocarbon
systems, cubic EoS (SRK/PR) are accurate and computationally fast.
When polar or associating species are present, CPA or SAFT-family
models are required.

In [5]:
# Demonstrate: water content in methane — SRK vs CPA
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil

T_test = 273.15 + 25.0
P_test = 50.0

results = {}
for label, cls, mr in [("SRK", SystemSrkEos, "classic"), ("CPA", SystemSrkCPAstatoil, 10)]:
    f = cls(T_test, P_test)
    f.addComponent("methane", 0.99)
    f.addComponent("water", 0.01)
    if isinstance(mr, int): f.setMixingRule(mr)
    else: f.setMixingRule(mr)
    f.setMultiPhaseCheck(True)
    ops = ThermodynamicOperations(f)
    ops.TPflash()
    f.initProperties()
    # Water in gas phase (ppm)
    y_water = float(f.getPhase("gas").getComponent("water").getx())
    results[label] = y_water * 1e6

print("Water content in methane gas at 25 C, 50 bar:")
for m, v in results.items():
    print(f"  {m}: {v:.0f} ppm")
print(f"CPA gives different (typically more accurate) water content prediction")

Water content in methane gas at 25 C, 50 bar:
  SRK: 598 ppm
  CPA: 763 ppm
CPA gives different (typically more accurate) water content prediction


**Answer:** This exercise demonstrates that EoS selection has practical
consequences. For water content in gas, the CPA EoS gives significantly
different (and more accurate) predictions than SRK because it properly
accounts for water's hydrogen bonding. The wrong EoS choice can lead
to hydrate problems or unnecessary dehydration costs.